<img src="../Imagenes/logo-final-ap.png" width="85" align="left"/>

# <span style="color:#bd232b">Aprendizaje Profundo</span>

[https://github.com/AprendizajeProfundo](https://github.com/AprendizajeProfundo)

# <span style="color:#174ea6">Cuaderno 102A · Deep Q-Learning: cómo una máquina aprende de sus propias experiencias</span>

**Sesión 1 · De las decisiones tabulares al aprendizaje profundo y Hydro-Lab**

**Autores:** Alvaro Montenegro · Daniel Montenegro · Pilar Montenegro  
**Docente:** Alvaro Montenegro, [ammontenegrod@gmail.com](mailto:ammontenegrod@gmail.com),  
**Institución:** Universidad de los Andes - Ecopetrol  
**Edición docente:** 102 · Octubre de 2026  
**Licencia del material original de esta edición:** MIT (véase sección final).  

> **Pregunta rectora.** ¿Cómo puede un agente aprender el valor de sus decisiones sin conocer las probabilidades de transición, y qué cambia cuando la tabla de valores debe ser sustituida por una red neuronal?


---

## Objetivos de aprendizaje

Al finalizar esta lección, el estudiante podrá explicar cómo un agente genera experiencias al interactuar con un ambiente, cómo esas experiencias se convierten en datos de entrenamiento y cómo una red neuronal aprende a estimar el valor de las acciones.

También comprenderá el papel de la ecuación de Bellman, el Replay Buffer, las redes Online y Target, la exploración y la retropropagación. Finalmente, podrá distinguir Deep Q-Learning de otros enfoques de aprendizaje por refuerzo, particularmente PPO.

La pregunta que guiará toda la lección es:

**¿Cómo puede una máquina aprender a tomar buenas decisiones si nadie le proporciona las respuestas correctas?**

---

# 1. Un robot que no sabe qué hacer

Imaginemos un pequeño robot situado dentro de un laberinto.

El robot tiene una misión: encontrar una estación de recarga.

No conoce el camino. Nadie le ha entregado un mapa con la ruta correcta. Tampoco tiene instrucciones que indiquen qué movimiento debe ejecutar en cada lugar.

Sin embargo, dispone de tres capacidades fundamentales:

1. Puede observar el ambiente.
2. Puede ejecutar acciones.
3. Puede recibir recompensas o penalizaciones.

El robot puede avanzar, retroceder o girar. Si choca contra una pared, recibe una penalización. Si llega a la estación de recarga, recibe una recompensa importante.

Al comienzo, sus movimientos pueden ser prácticamente aleatorios.

Pero después de muchas experiencias podría aprender a evitar obstáculos, reconocer situaciones favorables y encontrar caminos que conducen a la meta.

Esta es la idea fundamental del **aprendizaje por refuerzo**, conocido como *Reinforcement Learning* o RL.

En lugar de aprender a partir de ejemplos con respuestas correctas, el agente aprende a partir de las consecuencias de sus propias acciones.



<img src="../Imagenes/robot_ambiente.png" width="800"/>

*Figura 102A.1. El robot y su ambiente. Elaboración propia con ayuda de chatGPT.*



<img src="../Imagenes/gif_01_primeros_movimientos.gif" width="800"/>

*Figura 102A.2. El robot no recibe instrucciones sobre el camino correcto; recibe información sobre las consecuencias de sus movimientos. Elaboración propia con ayuda de chatGPT.*

---

# 2. El ciclo fundamental del aprendizaje por refuerzo

Vamos a estudiar qué sucede durante una sola interacción.

Supongamos que el robot se encuentra frente a una pared.

Esta situación constituye su **estado**, que representamos mediante la letra $s$.

El estado contiene la información disponible para tomar una decisión. En un ambiente sencillo puede ser la posición del robot. En otro más complejo podría incluir distancias a obstáculos, velocidad, orientación o imágenes de una cámara.

El robot selecciona una acción $a$, por ejemplo, girar a la izquierda.

El ambiente responde produciendo tres elementos:

- Una recompensa $r$.
- Un nuevo estado $s'$.
- Una indicación de si el episodio ha terminado.

La interacción completa puede representarse mediante:

$$
(s_t,a_t,r_t,s_{t+1},d_t)
$$

Aquí:

- $s_t$ es el estado antes de actuar.
- $a_t$ es la acción ejecutada.
- $r_t$ es la recompensa recibida.
- $s_{t+1}$ es el estado resultante.
- $d_t$ indica si ocurrió una terminación del episodio.

Esta colección de cinco elementos recibe el nombre de **transición**.

Una transición es la unidad básica de experiencia utilizada por DQN.


<img src="../Imagenes/gif_02_una_transicion.gif" width="800"/>

*Figura 102A.3. los cinco componentes de una transición.*




---

# 3. ¿Qué es un episodio?

Un episodio es una secuencia de interacciones que comienza en un estado inicial y termina cuando se cumple una condición de finalización.

Por ejemplo, nuestro robot puede terminar un episodio cuando alcanza la estación de recarga o cuando su batería se agota.

Consideremos el siguiente episodio:

| Paso | Estado | Acción  | Recompensa | Siguiente estado | Terminado |
| ---- | ------ | ------- | ---------- | ---------------- | --------- |
| 1    | A      | Derecha | −1         | B                | No        |
| 2    | B      | Arriba  | −1         | C                | No        |
| 3    | C      | Derecha | −1         | D                | No        |
| 4    | D      | Arriba  | +100       | Meta             | Sí        |

Este episodio contiene cuatro transiciones.

Ahora imaginemos que el robot repite la experiencia 1.000 veces.

Cada episodio puede tener una duración diferente. Algunos terminarán rápidamente, otros incluirán muchos movimientos inútiles y otros alcanzarán la meta.

Después de numerosas interacciones tendremos miles de transiciones.

**Y aquí aparece una idea extraordinaria: el agente acaba de producir sus propios datos de entrenamiento.**


<img src="../Imagenes/tres_episodios_de_aprendizaje_en_laberinto.png" width="800"/>

*Figura 102A.4. Distinción entre un episodio completo de las transiciones individuales que lo componen.*


**Descripción artística:** El mismo laberinto, siempre con el robot comenzando desde A. Aparecen sucesivamente tres recorridos: uno rojo que termina en un obstáculo, uno amarillo con varios desvíos y uno verde que alcanza la meta.

A la derecha aparece un contador:

«Episodio 1», «Episodio 2», «Episodio 3».

Debajo, otro contador aumenta: «Transiciones almacenadas: 4, 11, 17...».

**Propósito:** Distinguir claramente un episodio completo de las transiciones individuales que lo componen.

---

# 4. El Replay Buffer: la memoria de experiencias

DQN utiliza una estructura denominada **Replay Buffer**.

Se trata de una memoria en la que almacenamos transiciones observadas durante la interacción con el ambiente.

Cada registro contiene:

$$
(s,a,r,s',d)
$$

Por ejemplo:

| Estado | Acción  | Recompensa | Siguiente estado | Terminado |
| ------ | ------- | ---------- | ---------------- | --------- |
| A      | Derecha | −1         | B                | No        |
| B      | Arriba  | −1         | C                | No        |
| C      | Derecha | −1         | D                | No        |
| D      | Arriba  | +100       | Meta             | Sí        |
| A      | Arriba  | −10        | A                | No        |
| A      | Derecha | −1         | B                | No        |

Aunque las experiencias procedan de episodios, el Replay Buffer convencional almacena transiciones individuales.

Esto tiene una consecuencia importante.

Para entrenar la red no necesitamos recuperar necesariamente un episodio completo. Podemos seleccionar aleatoriamente, por ejemplo, 32 transiciones procedentes de distintos momentos y episodios.

A ese conjunto se le denomina **minibatch**.

¿Por qué seleccionar experiencias aleatoriamente?

Porque las observaciones consecutivas suelen estar muy correlacionadas. Si el robot se aproxima a una pared, probablemente observará la misma pared durante varios pasos.

Entrenar solamente con esas experiencias consecutivas puede producir actualizaciones excesivamente dependientes de situaciones recientes.

El muestreo aleatorio ayuda a reducir ese problema.

Además, una experiencia puede reutilizarse varias veces durante el entrenamiento.

**El Replay Buffer no es una tabla de respuestas correctas. Es una colección de experiencias observadas.**


<img src="../Imagenes/DiagramaReplay_Buffer_entrenamiento.png" width="800"/>

*Figura 102A.5. Las experiencias se almacenan y reutilizan.*



---

# 5. ¿Qué queremos que aprenda la red neuronal?

Ahora debemos formular una pregunta esencial.

Si tenemos miles de transiciones, ¿qué queremos aprender a partir de ellas?

No buscamos simplemente predecir la siguiente posición del robot.

Queremos que aprenda **qué tan conveniente resulta ejecutar cada acción en cada estado**.

Para eso utilizamos la función Q:

$$
Q(s,a)
$$

La letra Q hace referencia a *Quality*, calidad.

El valor Q representa el retorno esperado cuando el agente ejecuta una acción en un estado y después continúa siguiendo una determinada política.

Una política es, sencillamente, una regla para seleccionar acciones.

En Q-Learning buscamos aproximar la función óptima:

$$
Q^\*(s,a)
$$

Esta función expresa el máximo retorno esperado que puede conseguirse al ejecutar primero la acción $a$ en el estado $s$ y actuar óptimamente después.

Por ejemplo:

| Acción               | Valor Q estimado |
| -------------------- | ---------------- |
| Avanzar              | −10              |
| Girar a la izquierda | +8               |
| Girar a la derecha   | +3               |

Si el robot quiere aprovechar lo aprendido, seleccionará girar a la izquierda.

¿Por qué?

Porque es la acción con mayor valor Q estimado.

Sin embargo, estos números no representan necesariamente recompensas inmediatas. Incluyen la valoración de las recompensas futuras.


<img src="../Imagenes/Robot_en_laberinto_ valores_Q.png" width="800"/>

*Figura 102A.6. Una misma situación puede tener distintos valores Q asociados a las acciones disponibles.*

---

# 6. ¿Por qué importan las recompensas futuras?

Imaginemos que nuestro robot dispone de dos caminos.

El primero conduce a una recompensa de 5 después de dos pasos.

El segundo conduce a una recompensa de 20 después de cinco pasos.

¿Debería escoger siempre el segundo?

No necesariamente.

En aprendizaje por refuerzo utilizamos un factor de descuento, representado por:

$$
0\leq\gamma\leq1
$$

El factor $\gamma$ determina cuánto peso asignamos a las recompensas futuras.

El retorno descontado se expresa mediante:

$$
G_t=r_t+\gamma r_{t+1}+\gamma^2r_{t+2}+\cdots
$$

Cuando $\gamma$ es cercano a cero, las recompensas inmediatas dominan.

Cuando es cercano a uno, las recompensas futuras conservan mayor importancia.

Para nuestro ejemplo, suponiendo que los premios se reciben al completar los pasos indicados y no hay recompensas intermedias:

$$
V_A=5\gamma^2
$$

$$
V_B=20\gamma^5
$$

Con $\gamma=0.9$:

$$
V_A=5(0.9)^2=4.05
$$

$$
V_B=20(0.9)^5\approx11.81
$$

El segundo camino tiene mayor valor descontado.

Pero con $\gamma=0.2$:

$$
V_A=0.20
$$

$$
V_B=0.0064
$$

Ahora resulta preferible el primero.

El descuento permite equilibrar beneficios inmediatos y futuros.

<img src="../Imagenes/gif_05_dos_caminos.gif" width="800"/>

*Figura 102A.7.Un robot y dos tasas de descuento distintas*

---

# 7. La ecuación de Bellman: aprender del futuro

Llegamos al corazón de Q-Learning.

Queremos actualizar nuestras estimaciones utilizando las recompensas observadas y las estimaciones de lo que ocurrirá después.

La regla clásica de actualización es:

$$
Q(s,a)\leftarrow Q(s,a)+
\alpha\left[
r+\gamma\max_{a'}Q(s',a')-Q(s,a)
\right]
$$

Aunque parezca complicada, contiene una idea muy sencilla:

**Tomamos nuestra estimación anterior y la corregimos según lo que la experiencia nos acaba de enseñar.**

El término:

$$
r+\gamma\max_{a'}Q(s',a')
$$

es el objetivo de aprendizaje para una transición no terminal.

El término:

$$
\delta=r+\gamma\max_{a'}Q(s',a')-Q(s,a)
$$

es el error de diferencia temporal, conocido como *Temporal Difference Error*.

Veámoslo numéricamente.

Supongamos:

- Valor Q actual: 10.
- Recompensa recibida: 2.
- Mejor valor Q del estado siguiente: 15.
- Factor de descuento: 0.9.
- Tasa de aprendizaje: 0.1.

Primero calculamos el objetivo:

$$
y=2+0.9(15)=15.5
$$

El agente creía que su acción valía 10, pero la nueva experiencia sugiere 15.5.

Por tanto:

$$
\delta=15.5-10=5.5
$$

Corregimos la estimación:

$$
Q_{\text{nuevo}}=10+0.1(5.5)
$$

$$
\boxed{Q_{\text{nuevo}}=10.55}
$$

El agente acaba de modificar su estimación utilizando una experiencia.

En Q-Learning tabular se actualiza directamente el valor almacenado.

En Deep Q-Learning, en cambio, modificamos los parámetros de una red neuronal.

Esa diferencia es fundamental.


<img src="../Imagenes/Actualizacion_Q_learning_paso_a_paso.png" width="800"/>

*Figura 102A.8. Aprender no significa necesariamente reemplazar una predicción por el objetivo, sino corregirla gradualmente.*

---

# 8. ¿Por qué necesitamos una red neuronal?

En un ambiente pequeño podríamos almacenar todos los valores Q en una tabla.

Por ejemplo:

| Estado | Arriba | Derecha | Izquierda |
| ------ | ------ | ------- | --------- |
| A      | 2      | 8       | 3         |
| B      | 10     | 6       | 3         |
| C      | 4      | 7       | 9         |

Pero imaginemos que nuestro robot utiliza una cámara.

Cada imagen puede contener millones de valores posibles.

El número de estados diferentes puede ser inmenso.

No resulta viable construir una tabla que contenga todos los estados posibles.

La solución consiste en utilizar una red neuronal que aproxime la función Q.

La representamos mediante:

$$
Q_\theta(s,a)
$$

Donde $\theta$ representa los parámetros de la red.

En un DQN convencional con acciones discretas, la red recibe el estado y produce un valor Q para cada acción disponible.

Por ejemplo:

**Entrada:** Estado A.

**Salidas:**

- Q(A, Arriba) = 2.
- Q(A, Derecha) = 8.
- Q(A, Izquierda) = 3.

La red aprende una función capaz de producir estimaciones para estados que no necesariamente aparecieron de manera idéntica durante el entrenamiento.

Esa capacidad de generalización es una de las principales ventajas de utilizar redes neuronales.


<img src="../Imagenes/Red neuronal_Q_y_valores_de_accion.png" width="800"/>

*Figura 102A.9. La red recibe un estado y produce simultáneamente estimaciones para las acciones discretas.*

---

# 9. ¿Cómo se convierte una experiencia en un dato de entrenamiento?

Este es probablemente el concepto más importante de toda la lección.

En aprendizaje supervisado convencional tenemos ejemplos:

$$
(x_i,y_i)
$$

Donde $y_i$ es la respuesta conocida.

En DQN no conocemos directamente el valor Q correcto.

Tenemos experiencias:

$$
(s_i,a_i,r_i,s'_i,d_i)
$$

Por tanto, debemos construir los objetivos de entrenamiento.

Supongamos que extraemos esta transición del Replay Buffer:

$$
(A,\text{Derecha},-1,B,0)
$$

Esto significa que el agente estaba en A, ejecutó Derecha, recibió −1 y terminó en B, sin finalizar el episodio.

Supongamos que nuestra red de referencia estima:

$$
Q(B,\text{Arriba})=10
$$

$$
Q(B,\text{Derecha})=6
$$

$$
Q(B,\text{Izquierda})=3
$$

El mejor valor futuro estimado es 10.

Con $\gamma=0.9$:

$$
y=-1+0.9(10)
$$

$$
\boxed{y=8}
$$

Acabamos de construir un objetivo de entrenamiento.

Ahora consultamos la red que estamos entrenando:

$$
Q_\theta(A,\text{Derecha})=5
$$

Nuestra red predice 5, pero el objetivo construido es 8.

Utilizando error cuadrático:

$$
L(\theta)=(8-5)^2=9
$$

Mediante retropropagación, modificamos los parámetros de la red para reducir esta pérdida.

Aquí aparece una diferencia esencial respecto del aprendizaje supervisado tradicional.

**El valor objetivo no procede de una etiqueta externa conocida. Se construye combinando una recompensa observada con una estimación del valor futuro.**

Esta técnica se denomina *bootstrapping*.


<img src="../Imagenes/Como_aprende_un_DQN_paso_a_paso.png" width="800"/>

*Figura 102A.9. La etiqueta utilizada por DQN se calcula a partir de la experiencia y de una estimación.*

---

# 10. ¿Qué ocurre cuando termina el episodio?

Supongamos que el robot llega a la meta.

La última transición es:

$$
(D,\text{Arriba},100,\text{Meta},1)
$$

Como el episodio ha terminado, no añadimos recompensas futuras.

El objetivo es simplemente:

$$
\boxed{y=100}
$$

Ahora la red puede aprender que ejecutar Arriba desde D tiene un valor elevado.

Posteriormente, al entrenar con una transición que conduce desde C hasta D, el valor elevado de D puede influir en el objetivo correspondiente a C.

Y después puede influir en estados todavía anteriores.

De esta manera, la información sobre una recompensa final se propaga gradualmente hacia decisiones anteriores.

Esta es una de las propiedades más importantes del aprendizaje por diferencia temporal.

**Una recompensa obtenida al final de un recorrido puede terminar influyendo en decisiones tomadas mucho antes.**

En implementaciones modernas conviene distinguir entre una terminación real y una interrupción por límite de tiempo. Si el episodio se interrumpe artificialmente, puede ser apropiado conservar el término de valor futuro.


<img src="../Imagenes/Propagacion_de_valores_en_aprendizaje_por_refuerzo.png" width="800"/>

*Figura 102A.10. Cómo el aprendizaje de una recompensa final puede afectar estados anteriores, sin sugerir que todas las actualizaciones ocurren automáticamente en una sola pasada.*

---

# 11. Entrenamiento mediante minibatches

Hasta ahora hemos utilizado una sola transición.

En la práctica, normalmente entrenamos utilizando pequeños conjuntos de transiciones seleccionadas del Replay Buffer.

Supongamos que elegimos un minibatch de 32 experiencias.

Cada experiencia tiene su propio estado, acción, recompensa, siguiente estado e indicador de terminación.

Para cada una construimos un objetivo:

$$
y_i=r_i+\gamma(1-d_i)\max_{a'}Q_{\bar\theta}(s'_i,a')
$$

El símbolo $\bar\theta$ representa los parámetros de la red Target, que explicaremos enseguida.

La red principal produce una estimación para la acción efectivamente ejecutada:

$$
\hat y_i=Q_\theta(s_i,a_i)
$$

Aunque la red genere un valor para todas las acciones, la pérdida correspondiente a cada transición se calcula utilizando el valor de la acción que aparece en esa transición.

Una función de pérdida posible es:

$$
L(\theta)=\frac{1}{N}\sum_{i=1}^{N}(y_i-\hat y_i)^2
$$

Con $N=32$, obtenemos una pérdida promedio para el minibatch.

A continuación calculamos sus gradientes respecto de los parámetros de la red y ejecutamos un paso de optimización.

En implementaciones prácticas también es frecuente utilizar la pérdida de Huber, menos sensible que el error cuadrático a errores extremadamente grandes.


<img src="../Imagenes/entrenamiento_con_minibatches.png" width="800"/>

*Figura 102A.11. Distinguir el entrenamiento con una transición del entrenamiento con un minibatch.*



---

# 12. Dos redes neuronales: Online y Target

Existe un problema delicado.

Estamos entrenando una red utilizando objetivos que dependen de estimaciones de valores Q.

Si utilizáramos continuamente la misma red para producir las predicciones y los objetivos, ambos podrían cambiar simultáneamente de manera inestable.

Es parecido a intentar acertar a un blanco que se desplaza cada vez que realizamos un disparo.

DQN utiliza dos redes.

La **Online Network** es la red cuyos parámetros actualizamos mediante descenso por gradiente.

La **Target Network** se utiliza para calcular los valores futuros de referencia.

La Target Network mantiene sus parámetros fijos durante un intervalo de entrenamiento.

Periódicamente copiamos los parámetros de la Online Network a la Target Network.

También existen variantes que actualizan gradualmente los parámetros de referencia.

Así, los objetivos no cambian tan rápidamente como las predicciones de la red principal.

Este mecanismo mejora la estabilidad del entrenamiento, aunque no garantiza por sí solo la convergencia.


<img src="../Imagenes/Ciclo_redes_Online_target.png" width="800"/>

*Figura 102A.12. Una red aprende continuamente mientras la otra proporciona una referencia temporalmente estable.*


---

# 13. Exploración y explotación

Nuestro robot enfrenta un dilema.

Si siempre selecciona la acción con mayor Q estimado, puede quedarse atrapado en una estrategia mediocre.

Quizás exista un camino mejor que todavía no ha descubierto.

Pero si actúa siempre al azar, tampoco aprovechará lo que ha aprendido.

Este problema se conoce como el dilema entre **exploración y explotación**.

Una estrategia sencilla es $\varepsilon$-greedy.

Con probabilidad $\varepsilon$, seleccionamos una acción aleatoria.

Con probabilidad $1-\varepsilon$, seleccionamos una acción con el mayor valor Q estimado.

$$
a_t=
\begin{cases}
\text{acción aleatoria}, & \text{con probabilidad }\varepsilon\\
\arg\max_a Q_\theta(s_t,a), & \text{con probabilidad }1-\varepsilon
\end{cases}
$$

Por ejemplo, con $\varepsilon=0.2$:

- En el 20 % de las decisiones se utiliza el mecanismo de exploración aleatoria.
- En el 80 % se selecciona una acción de máximo Q estimado.

Es importante observar que una acción aleatoria también puede coincidir con la acción de mayor Q.

Habitualmente, comenzamos con un valor relativamente alto de $\varepsilon$ y lo reducimos progresivamente.

### [ESPACIO PARA GIF 11 — Explorar o explotar]

**Descripción artística:** Un robot frente a dos caminos.

Sobre él aparece una moneda simplificada con dos caras: «Explorar» y «Explotar».

Cuando aparece «Explorar», el robot toma una dirección aleatoria.

Cuando aparece «Explotar», selecciona el camino resaltado en verde con mayor Q estimado.

Un indicador muestra cómo $\varepsilon$ disminuye de 1.0 a 0.1.

**Propósito:** Mostrar la diferencia entre descubrir nuevas posibilidades y utilizar el conocimiento adquirido.


<img src="../Imagenes/Proceso epsilon-greedy_explorar_explotar.png" width="800"/>

*Figura 102A.13. La diferencia entre descubrir nuevas posibilidades y utilizar el conocimiento adquirido.*


---

# 14. El algoritmo completo de Deep Q-Learning

Ya tenemos todas las piezas.

El entrenamiento puede describirse mediante el siguiente procedimiento.

**Inicialización**

Creamos una Online Network con parámetros $\theta$.

Creamos una Target Network con parámetros iniciales iguales:

$$
\bar\theta=\theta
$$

Creamos un Replay Buffer inicialmente vacío.

Definimos el factor de descuento, la estrategia de exploración, el optimizador y los demás hiperparámetros.

**Interacción y entrenamiento**

1. Observamos el estado actual.
2. Seleccionamos una acción mediante $\varepsilon$-greedy.
3. Ejecutamos la acción en el ambiente.
4. Recibimos recompensa, siguiente estado e información de terminación.
5. Almacenamos la transición en el Replay Buffer.
6. Cuando tenemos suficientes experiencias, extraemos un minibatch aleatorio.
7. Calculamos los objetivos utilizando la Target Network.
8. Calculamos las predicciones utilizando la Online Network.
9. Evaluamos la función de pérdida.
10. Actualizamos los parámetros de la Online Network mediante retropropagación.
11. Actualizamos periódicamente la Target Network.
12. Continuamos interactuando y entrenando.

Cuando termina un episodio, reiniciamos el ambiente y comenzamos otro.

Por tanto, **no es necesario generar primero todos los episodios y entrenar después**.

En el DQN convencional, la generación de experiencias y el entrenamiento suelen intercalarse.

### [ESPACIO PARA GIF 12 — El ciclo completo]

**Descripción artística:** Diagrama circular con cinco cajas grandes:

«Ambiente» → «Experiencias» → «Replay Buffer» → «Entrenamiento DQN» → «Política mejorada».

Una flecha conecta la política mejorada con el ambiente, cerrando el ciclo.

Dentro de «Entrenamiento DQN» aparecen dos redes pequeñas, Online y Target.

**Animación:** Una ficha azul recorre lentamente el circuito. Cada vuelta representa una nueva interacción y actualización.

**Propósito:** Integrar todos los conceptos de la lección en una sola imagen.

<img src="../Imagenes/GIF_12_ciclo_completo_DQN.gif" width="800"/>

*Figura 102A.14. Integración de todos los conceptos.*

---

# 15. ¿Podemos entrenar con un buffer generado por un agente aleatorio?

Sí, en principio.

Imaginemos que ejecutamos un agente con acciones casi completamente aleatorias y almacenamos 100.000 transiciones.

Después podríamos utilizar esas transiciones para entrenar una red DQN.

Esto es posible porque DQN pertenece a la familia de algoritmos *off-policy*.

Significa que puede aprender a partir de experiencias generadas por una política diferente de la política que está intentando mejorar.

Pero existe una limitación importante.

Si el agente aleatorio nunca visitó ciertas regiones del ambiente, el buffer no contendrá experiencias de esas regiones.

La red tendrá dificultades para aprender correctamente qué sucede allí.

Además, si dejamos de interactuar con el ambiente y entrenamos exclusivamente sobre un conjunto fijo de transiciones, entramos en el ámbito del *offline reinforcement learning*.

En ese contexto, el uso ingenuo de objetivos Bellman puede producir sobreestimaciones y errores asociados con acciones insuficientemente representadas en los datos.

Por eso, aunque DQN pueda reutilizar experiencias antiguas, la cobertura del ambiente y la calidad de los datos siguen siendo fundamentales.

---

# 16. ¿Qué aprende realmente la red neuronal?

Esta pregunta merece especial atención.

La red no aprende necesariamente un mapa explícito del laberinto.

Tampoco memoriza simplemente una lista de instrucciones.

Aprende parámetros que permiten estimar valores Q a partir de los estados.

Internamente, el estado atraviesa varias transformaciones:

$$
s\longrightarrow h_1\longrightarrow h_2
\longrightarrow\cdots\longrightarrow Q(s,\cdot)
$$

Las variables $h_1,h_2,\ldots$ representan las activaciones o representaciones internas de las capas.

Durante el entrenamiento, estas representaciones pueden organizar información útil para predecir el valor de las acciones.

Por ejemplo, podrían reflejar configuraciones de obstáculos, distancias relevantes o situaciones que requieren decisiones similares.

Pero debemos evitar una interpretación excesiva.

Que una red aprenda a seleccionar acciones convenientes no significa automáticamente que haya construido una representación causal del ambiente.

Tampoco significa que posea una memoria autobiográfica o que comprenda el mundo de la misma manera que un organismo biológico.

Lo que sí podemos afirmar es que sus parámetros se ajustan para mejorar la aproximación de una función de valor.

El estudio experimental de las representaciones internas permite investigar qué información ha aprendido realmente la red.

---

# 17. ¿En qué se diferencia DQN de PPO?

DQN no es el único algoritmo de aprendizaje por refuerzo profundo.

Una alternativa ampliamente utilizada es PPO, *Proximal Policy Optimization*.

La diferencia conceptual principal es la siguiente.

**DQN aprende valores de acciones.**

Después selecciona las acciones utilizando esos valores, junto con algún mecanismo de exploración.

**PPO aprende directamente una política parametrizada**, normalmente acompañada de una función de valor utilizada durante el entrenamiento.

| Característica      | DQN                               | PPO                                    |
| ------------------- | --------------------------------- | -------------------------------------- |
| Enfoque principal   | Basado en valores                 | Gradiente de política                  |
| Qué aprende         | Función Q                         | Política y habitualmente valor         |
| Acciones            | Habitualmente discretas           | Discretas o continuas                  |
| Experiencias        | Reutiliza experiencias del buffer | Utiliza datos recientes de la política |
| Tipo de aprendizaje | Off-policy                        | On-policy                              |
| Estabilización      | Replay Buffer y Target Network    | Objetivo recortado y otras técnicas    |

DQN es especialmente adecuado para comprender cómo las recompensas futuras pueden transformarse en estimaciones de valor.

PPO permite estudiar directamente cómo se modifica una distribución de probabilidad sobre las acciones.

Ambos algoritmos son herramientas importantes, pero responden a estrategias de aprendizaje diferentes.

---

# 18. Limitaciones de Deep Q-Learning

DQN fue un avance fundamental, pero no resuelve todos los problemas del aprendizaje por refuerzo.

Entre sus principales limitaciones encontramos:

**Sobreestimación de valores.** El operador máximo utilizado en Bellman puede introducir sesgos positivos cuando las estimaciones contienen errores. Double DQN fue desarrollado para reducir este problema.

**Exploración insuficiente.** Una política $\varepsilon$-greedy puede resultar poco eficiente cuando las recompensas son escasas o requieren secuencias largas de decisiones.

**Inestabilidad del entrenamiento.** La combinación de aproximación funcional, aprendizaje off-policy y objetivos construidos mediante bootstrapping puede producir dificultades de convergencia.

**Acciones continuas.** El DQN convencional requiere calcular o aproximar el máximo sobre las acciones disponibles, lo cual resulta especialmente conveniente cuando estas son discretas.

**Dependencia de la recompensa.** El agente optimiza aquello que la función de recompensa incentiva. Una recompensa mal diseñada puede conducir a comportamientos indeseados.

Existen numerosas extensiones, entre ellas Double DQN, Dueling DQN, Prioritized Experience Replay y Rainbow.

---

# 19. Ejercicio integrador

Vamos a comprobar si hemos comprendido el mecanismo.

Un robot se encuentra en el estado A y ejecuta la acción Derecha.

Recibe una recompensa de −2 y llega al estado B.

El episodio no termina.

La Target Network estima los siguientes valores para B:

| Acción    | Valor Q |
| --------- | ------- |
| Arriba    | 12      |
| Derecha   | 7       |
| Izquierda | 4       |

La Online Network estima:

$$
Q_\theta(A,\text{Derecha})=5
$$

Utilizamos:

$$
\gamma=0.9
$$

**Pregunta 1.** ¿Cuál es el objetivo Bellman?

**Pregunta 2.** ¿Cuál es el error de diferencia temporal?

**Pregunta 3.** ¿Cuál es la pérdida cuadrática para esta transición?

**Pregunta 4.** ¿Qué parámetros se modifican durante el paso de entrenamiento?

**Pregunta 5.** ¿La transición permite afirmar que el valor verdadero de ejecutar Derecha en A es exactamente igual al objetivo calculado?

### Solución

El objetivo es:

$$
y=-2+0.9(12)=8.8
$$

El error de diferencia temporal es:

$$
\delta=8.8-5=3.8
$$

La pérdida cuadrática es:

$$
L=(8.8-5)^2=14.44
$$

Durante el entrenamiento se actualizan los parámetros de la Online Network. La Target Network permanece fija hasta su siguiente actualización programada.

Finalmente, el valor 8.8 no es necesariamente el valor verdadero de la acción.

Es un objetivo construido utilizando una recompensa observada y una estimación del valor futuro.

Esta última distinción es esencial.

---

# 20. Síntesis final: de la experiencia al aprendizaje

Podemos resumir todo el proceso en una secuencia conceptual.

El agente interactúa con el ambiente.

Cada interacción produce una transición.

Las transiciones se almacenan en el Replay Buffer.

Durante el entrenamiento, seleccionamos minibatches de experiencias.

Para cada transición, construimos un objetivo utilizando Bellman y las estimaciones de la Target Network.

La Online Network produce sus propias predicciones.

Comparamos las predicciones con los objetivos.

La función de pérdida cuantifica las discrepancias.

Mediante retropropagación, modificamos los parámetros de la Online Network.

La red actualizada influye en las próximas decisiones del agente.

Esas decisiones producen nuevas experiencias.

Y el ciclo continúa.

La idea más importante de toda la lección es esta:

**El Replay Buffer almacena hechos observados, pero los objetivos Q utilizados para entrenar la red son estimaciones construidas a partir de esos hechos y de predicciones del propio sistema.**

En aprendizaje supervisado tradicional, las etiquetas suelen estar disponibles de antemano.

En DQN, los objetivos de entrenamiento evolucionan a medida que cambia el conocimiento estimado del agente.

Por eso el aprendizaje por refuerzo profundo es especialmente interesante: el agente participa en la producción de sus experiencias y utiliza esas experiencias para modificar su comportamiento futuro.

---

## Para recordar

Al terminar esta lección debemos poder distinguir claramente cinco conceptos:

- **Experiencia:** lo que sucedió cuando el agente ejecutó una acción.
- **Replay Buffer:** la memoria que almacena las transiciones observadas.
- **Función Q:** la estimación del valor de ejecutar una acción en un estado.
- **Objetivo Bellman:** la referencia calculada para entrenar esa estimación.
- **Red neuronal:** el modelo parametrizado que aprende a aproximar la función Q.

La diferencia entre estos cinco elementos es la clave para comprender Deep Q-Learning.

## Próxima lección

**Profundización opcional: Deep Q-Learning por dentro, del minibatch a los gradientes.**

Seguiremos una actualización real de entrenamiento con cuatro estados, tres acciones y una pequeña red neuronal.

Veremos cómo se representan los estados como tensores, cómo se selecciona el valor Q de la acción ejecutada, cómo se calculan los objetivos Bellman para todo el minibatch y cómo la retropropagación modifica los pesos.

El objetivo será comprender el algoritmo hasta el punto de poder implementarlo y diagnosticar sus errores, sin depender de una caja negra.

---

*Fin de la lección 102A.*

Esta versión conserva los espacios reservados para recursos visuales, con especificaciones de composición, animación y objetivo pedagógico. Todos están pensados para producirse con una misma identidad visual minimalista y poder incorporarse después a un manual, una presentación o una aplicación educativa.

## Laboratorio numérico opcional · Verificar Bellman

Este pequeño cálculo reproduce el ejercicio integrador anterior. No requiere bibliotecas externas ni modifica el simulador Hydro-Lab.

In [ ]:
recompensa = -2
gamma = 0.9
q_target_siguiente = [12, 7, 4]
q_online_actual = 5
objetivo = recompensa + gamma * max(q_target_siguiente)
error_td = objetivo - q_online_actual
perdida = error_td ** 2
print(f"Objetivo Bellman: {objetivo:.1f}")
print(f"Error TD: {error_td:.1f}")
print(f"Pérdida cuadrática: {perdida:.2f}")
assert abs(objetivo - 8.8) < 1e-9
assert abs(error_td - 3.8) < 1e-9
assert abs(perdida - 14.44) < 1e-9